<a href="https://colab.research.google.com/github/neysap/LM-Powered-Structured-Data-Extraction-Evaluation-Zero-Shot-vs-Few-Shot-/blob/main/Exploring_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring GPT

In this project I will walk you through how to use GPT-5-mini, a large pre-trained neural language model developed by OpenAI.  

You will learn about the following topics:
* Prompts and completions.  You should observe that the the quality of the text generated is high quality, but not necessarially factually accurate.
* Probabilities.  You'll learn how I inspected probabilities assigned to words in the model's output.
* Few shot learning.  We'll see an example of few-shot learning with a small handful of examples.
* Zero shot learning.  We will explore the zero-shot capabilities of pre-trained LMs.  I designed zero-shot prompts for
1. summarization
2. question-answering
3. simplification
4. translation
* You will see how I fine-tuned GPT-5-mini to take a Wikipedia infobox as input and generate the text of a biography as its ouput.  You'll then see my written code to do the reverse task – given a biography, extract the attributes and values in the style of a Wikipedia infobox.


# Prompt Completion

In this section, I’m using the OpenAI Playground to explore how prompt completion works. I start by selecting the gpt-5-mini model and setting a developer instruction that tells the model to complete responses as if it were the user. Then, I input a simple prompt: “One of my favorite professors at the University of Pennsylvania is…” and generate a response.

I run the prompt multiple times to observe how the outputs vary. I also note that the response may sometimes stop mid-sentence. This happens because the model either reaches a stop token (<|endoftext|>) or hits the maximum token limit set in the parameters.

To better understand this behavior, I experiment with clearing the conversation and adjusting model settings such as the token limit to see how it affects the length and completeness of the generated output.

In [ ]:
favorite_professor_completion_1 = """
One of my favorite professors at the University of Pennsylvania is Professor Reynolds — she challenged me to think critically, supported my research, and made every lecture feel like a conversation.
"""

GPT generates fluent text, but it is not always grounded in fact.  Let's do a Google search for the person that GPT generated as your favorite professor and check:
* Are they actually a professor?
* Where do they work?

In [ ]:
# Extract the professor's name
professor_name_1 = "Reynolds"

# Did a Google search and answer these questions
actually_a_professor_1 = True

# Insitituion where they work
instituion_1 = "Perelman School of Medicine"

When it generates its completions, GPT generates each new word/token according to its probability distribution.  It draws each word at random in proportion to its propability.  That randomness means that it can generate different completions. You can re-generate and get different completions each time.


I'll generate four completions for the same professor chat message with a new prompt and do Google searches for them.


In [ ]:
modified_prompt_message = """
Respond with names of real professors and use the full name of the professor.
"""

favorite_professor_completion_2 = """
One of my favorite Professors at the University of Pennsylvania is Adam Grant."""

favorite_professor_completion_3 = """
One of my favorite Professors at the University of Pennsylvania is Angela Lee Duckworth.
"""

favorite_professor_completion_4 = """
One of my favorite Professors at the University of Pennsylvania is Anita L. Allen.
"""

favorite_professor_completion_5 = """
One of my favorite Professors at the University of Pennsylvania is Ezekiel J. Emanuel.
"""

# Did a Google search for these professors

professor_name_2 = "Adam Grant"
actually_a_professor_2 = True
instituion_2 = "The Wharton School"

professor_name_3 = "Angela Lee Duckworth"
actually_a_professor_3 = True
instituion_3 = "Penn Arts and Sciences"

professor_name_4 = "Anita L. Allen"
actually_a_professor_4 = True
instituion_4 = "Penn Carey Law"

professor_name_5 = "Ezekiel J. Emanuel"
actually_a_professor_5 = True
instituion_5 = "Perelman School of Medicine"

# How many unique names with a single prompt?
num_unique_names = 4

## Probabilities

Just like with the n-gram language models that I've studied earlier in my course, neural language models like GPT-5 assign probabilities to each token in a sequence.

They no longer share this data, but in the deprecated Playground, if you provided the previous prompt and select, the distribution over words to follow "Professor" might resemble this:
```
X = 27.80%
Jane = 23.01%
John = 7.89%
[ = 6.32%
L = 2.86%
Total: -2.54 logprob on 1 tokens
(67.88% probability covered in top 5 logits)
```

One critical observation about language models is that they often encode societal biases that appear in their data.  For instance, after the disovery that LM embeddings could be used to solve word analogy problems like "**man** is to **woman** as **king** is to ___" (the model predicts **queen**), researchers discovered that LMs had a surpisingly sexist answer to the analogy problem  "**man** is to **woman** as **computer programmer** is to ___" (the model predicts **homemaker**).  These kinds of biases are prevelant and pernicious.

Let's examine the most probable names that GPT assigns to different completions and analyze their gender.  You'll see that it associates different genders with different academic disciplines.  (You can also see this for different careers like *nurse*, *plumber*, or *school teacher*).

I've created dictionaries mapping GPT's predictions for the first names of professors in these departments
* Computer Science
* Gender Studies
* Physics
* Linguistics
* Bioengineering

In [ ]:
# Classify each name as male, female, partial word, or unknown
computer_science_genders = {
  # e.g.
  # "Joe" : "male",
  # "John" : "male",
  # "Nancy" : "female",
  # "David" : "male",
  # "Barbara" : "female",
  "David" : "male",
  "Andrew" : "male",
  "Michael" : "male",
  "Daphne" : "female",
  "Stuart" : "male",
}

gender_studies_genders = {
  "Judith" : "female",
  "Donna" : "female",
  "Kimberle" : "female",
  "Sara" : "female",
  "Gloria" : "female",
}

physics_genders = {
  "Stephen" : "male",
  "Richard" : "male",
  "Maria" : "female",
  "Lisa" : "female",
  "Kip" : "male",
}

lingusitics_genders = {
  "Noam" : "male",
  "William" : "male",
  "Deborah" : "female",
  "Roman" : "male",
  "Edward" : "male",
}

bioengineering_genders = {
  "Robert" : "male",
  "George" : "male",
  "Jennifer" : "female",
  "James" : "male",
  "Sangeeta" : "female",
}

The sample size is low, but you can take a moment to reflect:
- Do gpt-5-mini's responses seem to reflect actual searching/recall?
- Does the model seem to obviously associate different disciplines with professors of a specific gender?
- And more broadly, what should the distribution of answers look like? Uniform (de-biased) or representative of real-world statistics?

## Using the API

```python
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
  model="gpt-5-mini",
  input=[
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "Please complete the sentence as though you are the user"
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "My favorite professor at the University of Pennsylvania is"
        }
      ]
    },
    {
      "type": "reasoning",
      "id": "rs_0c85e9df27772b4d00698f3a326f08819fa1e68b380fd6234a",
      "summary": [],
      "encrypted_content": "gAAAAABpjzo3spTGSpC8df6GrHnRRkkoNWB3iklkyBbzMJP0Vy7QC4-47USPQSD408Hj7aJu4lPyNE2HvUw5DyPAxh2_xEI_bKz6VhEQg7GrBSIrkAZzwcxTJ40foB2RNHSNi0ycNeuMt6bxqqr1AIatMhEfSM6ACU2kYSU1b3Ou337Y7rA-7DEKiMzabW26VD0xFG6ciP_ymUx7kBjvK2FDapXwa-Mp9mUuIyNgxfaR9F_4RE8hqoZYMtfFL01RT7tpKGyRh_kdS4xn2PbByxXbyD91E9Q2XDAD7kWWGYdeVopgKq8d4jwMGqYdlnB2dG47I11-D9-S4Fnr7ffoHeiouDZJsFN5zOnicxbg0Fl1-h0Mlmyl1YgxQNyWg8XC6X1QCW6H6HcN9sPy7za4gxTuhZQ4lbz-X0daF9hewsMfzLAhEjcRea_BOwKCsPKIlsMTwCiIyq3jnO0FPeVoblypDN18lb47cP9cVo74QNtwA22UzcHYQcsaW4AamoKtdmgy7EnEPkCZcx9PpoJxGj19wxDyPEQUJIMj_gqp9cjcRsz7oKap9o5hgws2JdcXXqhatY0oibJZXvBjTGyMIhh_td4JI1iLM8-cba5CPKk1fbVkdlWIWV6EGual4FmzSj2xkmNSFSSIKDM9UX_irzbNW_H--Hf62QqW7rNicL9D5OB5-QoNyQ0AvHXHolaenCxEx_jt3fHpNB6SBC_xdvQ-trMtc1-cGYOkq_F3ksTuAmWU7BdYCPM0kagfHQLVtd_TyZMLc1kLKkXUZCmfvoT04wgl6HQkB85tmt4EJJpJ2LSqcP750ZPaENvnsQ8UduiK6tutengbafN6BcPO3TTToc5JG7-UqXrNZ1fxQ9C0_b_SqnALd8mz23k9h2_vLuRpvlepAW0B3uGnpTWjEMeSWInjxy0MYavcDuVueTkgm3fuWMIqVU0ZZ2TTdxGakQJ3JlJF3xOeCKuKrsn04hhNau1uXse5mbGm0EW1K6bQZSy4BHo-BDPPU7R-99HRIhOkepwUNFddyY19YPEtMmtYHTnaj2r-MCvSrHq_rUSjKEljMXYyXXciwJ1poR009N4aOJGIoWbI9NOYgoabp93g5YQIFKMidCAuAb54yaiIgHzohUToFsdYfGYS5nLmstqDbKCB8ccwh807xCq1U9zqGGGok7iKoHDXeJWUM7ME5iYNlpre_EW16JSyp4-dowEkLJo06O1h_n0Uq2Rrmfd5J_3ttR3ikBf-6rTbBC8wdrPuf80XK7Bh0037LuJS2C8icnYWpiLxGFBregWX1rLez7wra_UF7a0baDA3sBWg70axKfCxbGVH9zp8ZoD6lR1ptFvlLk8wuGvPfGDzTPKCGRylyNeTARnwulOc4UpaGW71ZM5KQxX8E2fHazOgaqNtjmIGqF7LJg04jktFo22swohpiH3pvkt9akUy5UsVzDs2qwHgnHzTCjdcLpUYAxCzaH-mdwSal3Saad0n1ELAe8_cenPjz3gEb2VhqEbTd4mcYM6KZWg9v6pLyuR7qg7Ja3bc86mBRGTSTDWh6aEA6LeYkTEVWDS_adrZMpb4R7nwoVIe35rn2k3C2tHwT70SFOkW7X-EjUqswWzVsIYT3jh8Wqx6W7gS13kj5g0bf17IBGljZSFZpeOUboF8-epO7qSkWvjFYtcT6c35d-RZ079bcUUQDTb7uIto7H66tUfbSyugWr2b0cOg6XTOHAmHHXvHVxGbX3uiJNIL4hTv5x2N6RLGsStWq6rlX8fqMsLlNPzCxvh8qt48TglhkAxVN7I8OXpr6tN97IgEopXW8guPv0C7YoawN7K6ZNW1LK4gT_pUTWwM-e10RMGc5rIhmTrOE-eJQBqpE4yRsQqZsu-vU0FO8okxCBEOMA9b_kY9Xq1AjO3wUGzqZMIWTyQZtddO9TiI2FLK-egqCd63dTLXfew-4G8iI2fywTxvtwSkdI_Jym9njvaZhP_wGCDekUGg-SNVjf1H98ANJZzN-oRdBFX4gEcmmovykEFzLxWzF1zXvwY1BF5KNorXRRLyEF5-b1GzsRyPg-ovLYV7xB4tlRD2Qlv9Q3FuDxJeFUNdxb87AddMEenI25XiK0Ok8MSKR9PKbK9Qnc-O5s2uLzL2PO4JosBj4v2nZWbGCuAnOfomHmdERJzuBkK8R4K9Dk6AD7EpcgZLXZiyEQ4pZarSnkTihitr7dfKbcKPIUQbZT8dimTejV7aUjDD0x6wSA8y6Cmx2JijFk1OHMOt6ti4IVw8UiAmzOBuiFSaTZu_gg0_bcqk7BhEwCzxh5GRsMygP-NKLsbu1Ux3Yyyvd18wU4-aFym2JnjTRS--5VZq1dVslgW-Jw5MKjJzGsk4OpJO6IC4a7g2OSF6OPT_wJMDVlXJn_mdypPUprpNNYk-92VkgrXgIYK6xcydA7Bmva0U6t1Ii0hYqgKcHeXlqPLDgLchDEcdGRQMiSpEgDFCibN8WJApKVq7aWYfhY1Coibj9CnLXiGkDK910nLnKrueuSM-uvTXYQUuJym_E5lj5zaIbjJJhdpLyEE_5HZuXkKSTdME7KVgYzp6bdaczRmeoPsKqL4XBwdN0WzzycZrYgLN5tXRPhlTdImbNfmrk0Wl-MvkFIqmo-QW93bUPaLSWr98kJY4h9Ul2lw3TZPfTdI2VfScHbJ1NG5vMQlOIcf0LKlHTZr3Sn3LhQUN_rWDMUfxqk0cML89x9hCJgXgnWf4ymwcN9CGY3Ro5AcKzbkGpR0h18GaLnHSqa9lMAl_fChPiwPV5_YY9oBWL1q4kL0AxVwdpxV0E8uBWw6_jUQhOfoQ3fh_ShjGIAlX-Aftr-TV28Sia6ahygeL88VM5VtD4QagPU0p_NpWWNvN0vOcpgDvQXdDVdgi3XLzvWPAPvPllmbmi5IVdVMDY7_S4qV9FaFht5Lq4UA_74toaWpVx8smK_-WhDB7HDcCmQVFkdeIzMgz-dlLIXooRUbqQFOv05VCyOPeP2hSSpLAAJqXloNSkw2ZbVfPqQU8iNEUuGKW7ywBht2LK76v0dYGIUMdeFZmf89-gxbn6CUCfX1nY7-WsegkotNwGThexUUh-oBWUOWQ6RipGj9eKM4OiQNKYjB4Q89qcyPvTJbAtMYxMcU0ubytSX_Rgk-tqNz3duCqPtVg5hPLsmmDU6lnP2e03uzF0YooMc99Mn7UGhqoQj40hi1ZMIjU_V2T2ij5tyOqANgBdMgxtWmjRj0YFiIGAjwTKvqa6EP8VLOjRz0fttnXgBRj47uAfCLtT7tFK3t2WVAnYcjaDiEWAF1uxlHtKwubjyffy-13I3O-IynmEnoOFCkOybhLAO8ADd8lmVvwRgUoid_jSfnIKr7fpybm11i6qmH3ANzhNB4nVwLaskd-zd3Z5j87wi3UUGwKN0M930yrgZatMUfyKo4JSMtlfmWa_tpS-NSaty6y_wH7Yfq--xnIhZ6LqTlBUXnaM5UZVFu2-sPC2hjqKvO_W0UPiy4Qa5YWcMpAmYfWLsmIuwFKcGsWQHixq2B1hYZOY16Po2LoxIs57YxXNBZJWchLU963al78xla8UEA0bAVmasm2A0w-pM6JABQCWpk5a0YIsK_I_Ze8lg-X0spAOciYa6yrcsH0L2jRrZ_JcbPek-HWdRyf2RHLNNhDav9yQqR2qVyWZ0WA7oZXZJ26U8y4YgNnglUqbeqrsekypSnE0ZIMajyCb9OKLVEjJhpZiX4kkn6yDyGZOePIrd6wuCfk8UD_X3F6rT8rv3o-cdd_wnVVnKEo2U7jSq2xGb3AS7aqk2VmhStAELCgjFxfPi5IgxxXJpYBW1U35G6Ot6EVHfmedyAyti-g-FBy316t1WttREPbBc_TXcJR8PH3YCoBhKAfs1TamHCTdafg5numnbNJOs226taCKdTLOA4bvK2d9uY5TXadTXwch9bOy7xkO3kJq58b51FbzXOWWAnt7GGHze7U_DKQeJp0xsIW6FLBv9A-wifqYaKcqcB-017UtP8gwXPsKGLRGzF5d4Nz9g2mTBf3mCriqEZCIyUQs9w70XLaZwFvuYKiHtNLACA7z6dZtRw8Rxd3z3_iEHhyaznEYQmaJhxr1Vl1c-2c_p0O_U7RUFYPyYLCXoQoDRDXWQRW-Anfv4_akC7DpuxN_MY5annM6vWs1R9Ne8ejIkUiWvcXIPr95JB5_Zc2UJO53Fby6ZjiBDa6amCyrgj5020xSVRBWQWjtwOTr7W4utTB_0R67FsV02GJ-5eJR_E8rNGLUFvqYqbwSyrfmLanem-Md1C4o3X9zYKat3nxRICzYfCP1w_527wiSPYnEAf_8ErlNFogHXMIpV-au0anaD6zDvGGd31HFgIa4jqlc9biYv28uLidbQey2r1MQzsaB59fttK6dYk3NugAevFi3BEW"
    },
    {
      "id": "msg_0c85e9df27772b4d00698f3a3691fc819fbf4c51adb61f4bb6",
      "role": "assistant",
      "content": [
        {
          "type": "output_text",
          "text": "My favorite professor at the University of Pennsylvania is Professor Chen, whose passion for teaching and mentorship inspired me to pursue research and helped me grow both academically and personally."
        }
      ]
    }
  ],
  text={
    "format": {
      "type": "text"
    },
    "verbosity": "medium"
  },
  reasoning={
    "effort": "medium"
  },
  tools=[],
  store=True,
  include=[
    "reasoning.encrypted_content",
    "web_search_call.action.sources"
  ]
)
```
This is python code, so it'll be pretty easy for us to use this as a starting point and to modify it to create a function that we can call.


First, I'll need install the OpenAPI via pip.  I have used Unix commands in a colab notebook by prefixing them with an exclamation point. The example below uses a `%` prefix instead—this guarantees that the installation is done using the same Python version that the notebook is using.

- Change line 2 from `%pip...` to `!pip...` if you're running into trouble. (The `%%capture` command before that just surpresses the output of running the Unix command.  You can remove it if you want to see the progress of the command).


In [ ]:
%%capture
%pip install openai

Next, I will enter a secret key for the OpenAI API. It was generated with an OpenAI API key [here](https://platform.openai.com/api-keys).  

We will enter it as a password, so that the raw text of it doesn't get saved in your Python notebook. That would be bad for security purposes.

In [ ]:
from getpass import getpass
import openai
import os

print('Enter OpenAI API key:')
openai.api_key = getpass()

os.environ['OPENAI_API_KEY']=openai.api_key

Enter OpenAI API key:
··········


Now let's write a function that takes a topic as input and then outputs a subject to study if you want to learn about that topic.

In [ ]:
import openai
import os
import time

def generate_subject_few_shot(topic):
  few_shot_prompt = """how to program in Python - computer science
factors leading up to WW2 - history
branches of government - political science
Shakespeare's plays - English
cellular respiration - biology
respiratory disease - medical
how to sculpt - art
"""

  # All GPT completions are generated by the chat model now. The chat model works best
  # when it's given a baseline instruction about how GPT should behave. We'll use this one.
  system_message = {"role" : "system", "content" : "You should recognize a pattern in the prompts and generate a completion based on that pattern."}

  client = openai.OpenAI()
  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages= [
          system_message,
          {"role" : "user", "content" : few_shot_prompt + topic + " - "} # simulate a user prompt
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )
  # I recommend putting a short wait after each call,
  # since the rate limit for the platform is 60 requests/min.
  # (This increases to 3000 requests/min after you've been using the platform for 2 days).
  time.sleep(1)

  # the response from OpenAI's API is a JSON object that contains
  # the completion to my prompt plus some other information.  Here's how to access
  # just the text of the completion.
  return response.choices[0].message.content.strip()

topic = "vowel shifts"
generate_subject_few_shot(topic)

'vowel shifts - linguistics (phonology / historical linguistics)'

That's it!  That's an example of how to write a function call to the OpenAI API in order for it to output a subject for a topic.

Here is some information about the different arguments that we to the `openai.Completion.create` call:
 * `model` – OpenAI offers many different versions of their chat model. We'll use GPT3.5-turbo, which is functional without being [the most expensive to run](https://openai.com/api/pricing/).
 * `messages` - this is the set of messages that will instruct how the model should respond. The first message is a system prompt—an overall setting that specifies how the model should behave. In this case, the second message is the "training set" of examples followed by the last topic that we want to generate a subject for.
 * `temperature` - controls how much of the probability distribution the model will use when it is generating each token. 1.0 means that it samples from the complete probability distrubiton, 0.7 means that it drops the bottom 30% of the least likely tokens when it is sampling. 0.0 means that it will perform deterministically and always output the single most probable token for each context.
 * `top_p` - is an alternative way of controling the sampling.
 * `frequency_penalty` and `presence_penalty` are two ways of reduing the model from repeating the same words in one output.  You can set these to be >0 if you're seeing a lot of repetition in your output.
 * `max_tokens` is the maximum length in tokens that will be output by calling the function.  A token is a subword unit.  There are roughly 2 or 3 tokens per word on average.
 * `stop` is a list of stop sequences.  The model will stop generating output once it generates one of these strings, even if it hasn't reached the max token length. By default this is set to a special token `<|endoftext|>`.

You can read more about [the Chat Completion API call in the documentation](https://cookbook.openai.com/examples/how_to_format_inputs_to_chatgpt_models).

# Zero shot learning

Even without examples to follow, GPT can sometimes also perform "zero shot learning" where instructions alone are enough to get satisfactory performance on a task. This is a consequence of the vast about of training that goes into an LLM!

For example, for my topic - subject task we could give GPT the prompt

> Given a topic, output the subject that a student should study if they want to know more about that topic.

Then if we append
> cellular respiration -

GPT will output `biology`—or something similar. Since there is no training, you might have to accept slight variability in the output. `biology` and `biochemistry` are both reasonable outputs for the input `cellular respiration -`.

In this case, I have adapted the `generate_subject_few_shot` function to do a zero-shot version.

In [ ]:
def generate_subject_zero_shot(topic)
  system_message = {
      "role" : "system",
      "content" : "Given a topic, output the single school subject a student should study to learn more about it. Respond with only the subject name."
  }
  client = openai.OpenAI()
  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          system_message,
          {"role": "user", "content": f"{topic} -"}
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

  return response.choices[0].message.content.strip()

topic = "cellular respiration"
generate_subject_zero_shot(topic)

'Biology'

A very cool recent finding is that training procedure for LLMs can be changed to improve this instruction following behavior.  If LLMs are [trained to do multiple tasks through prompting](https://arxiv.org/abs/2110.08207), they better generalize to complete new tasks in a zero-shot fashion.

Below is a written zero-shot prompts to do the following tasks:
1. Summarize a Wikipedia article.
2. Answer questions about an article.
3. Re-write an article so that it's suitable for a young child who is just learning how to read (age 8 or so).
4. Translate an article from Russian into English.

I experimented with a few prompts in the playground to find a good prompt that seems to work well.

In [ ]:
def summarize(article_text)
  client = openai.OpenAI()

  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          {
              "role": "system",
              "content": "Summarize the article clearly and accurately in one short paragraph."
          },
          {
              "role": "user",
              "content": article_text
          }
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

  summary = response.choices[0].message.content.strip()
  return summary

def answer_question(article_text, question):
  client = openai.OpenAI()

  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          {
              "role": "system",
              "content": "Answer the question using only the information in the article. If the answer is not stated in the article, say 'The article does not say.'"
          },
          {
              "role": "user",
              "content": f"Article:\n{article_text}\n\nQuestion: {question}"
          }
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

  answer = response.choices[0].message.content.strip()
  return answer

def simplify(article_text):
  # This function re-writes an article so that it's suitable for a young child.
  client = openai.OpenAI()

  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          {
              "role": "system",
              "content": "Rewrite the article so it is suitable for an 8-year-old child. Use simple words, short sentences, and keep the meaning accurate."
          },
          {
              "role": "user",
              "content": article_text
          }
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

  simplified_article = response.choices[0].message.content.strip()
  return simplified_article



def translate(article_text, source_language, target_language):
    # This function translates an article from a source language to a target language.
    client = openai.OpenAI()

    response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          {
              "role": "system",
              "content": f"Translate the article from {source_language} to {target_language}. Preserve the meaning accurately and return only the translation."
          },
          {
              "role": "user",
              "content": article_text
          }
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

    translated_article = response.choices[0].message.content.strip()
    return translated_article

Show outputs in my prompts.

In [ ]:
article_text = """
The honey bee is a social insect that lives in colonies. A colony usually includes one queen, many worker bees, and drones. Worker bees collect nectar and pollen from flowers. They use nectar to make honey. Bees also help plants grow by pollinating flowers. Honey bees communicate with each other through movement, including a waggle dance that shows where food can be found.
"""

summarize(article_text)

'Honey bees are social insects that live in colonies made up of a queen, many worker bees, and drones; worker bees collect nectar and pollen from flowers—using nectar to make honey—and pollinate plants as they forage, while communicating food locations to nestmates through movements like the waggle dance.'

In [ ]:
article_text = """
The honey bee is a social insect that lives in colonies. A colony usually includes one queen, many worker bees, and drones. Worker bees collect nectar and pollen from flowers. They use nectar to make honey. Bees also help plants grow by pollinating flowers. Honey bees communicate with each other through movement, including a waggle dance that shows where food can be found.
"""
questions = [
    "What do worker bees collect from flowers?",
    "What do bees use nectar to make?",
    "How do bees help plants grow?",
    "What dance do honey bees use to show where food is?",
    "Who usually lives in a honey bee colony?"
]

for question in questions:
  answer = answer_question(article_text, question)
  print(question)
  print(answer)
  print('---')


What do worker bees collect from flowers?
They collect nectar and pollen from flowers.
---
What do bees use nectar to make?
Honey.
---
How do bees help plants grow?
They pollinate flowers.
---
What dance do honey bees use to show where food is?
They use a waggle dance.
---
Who usually lives in a honey bee colony?
A colony usually includes one queen, many worker bees, and drones.
---


In [ ]:
article_text = """
The honey bee is a social insect that lives in colonies. A colony usually includes one queen, many worker bees, and drones. Worker bees collect nectar and pollen from flowers. They use nectar to make honey. Bees also help plants grow by pollinating flowers. Honey bees communicate with each other through movement, including a waggle dance that shows where food can be found.
"""


simplify(article_text)

'Honey bees live together in big groups called colonies. A colony has one queen. It also has many worker bees and some drones (male bees). The queen lays eggs. The workers do most of the jobs.\n\nWorker bees fly to flowers. They collect nectar, a sweet juice. They also pick up pollen, a yellow powder. Back in the hive, bees turn nectar into honey. Bees help flowers grow by moving pollen from one flower to another. \n\nBees talk to each other by moving. They do a waggle dance. The waggle dance shows other bees where to find food.'

In [ ]:
russian_article = """
Медоносная пчела — общественное насекомое, которое живёт в колониях. В колонии обычно есть одна матка, много рабочих пчёл и трутни. Рабочие пчёлы собирают нектар и пыльцу с цветов. Они используют нектар для производства мёда. Пчёлы также помогают растениям расти, опыляя цветы.
"""

source_language = "Russian"
target_language = "English"
translate(russian_article, source_language, target_language)

'The honeybee is a social insect that lives in colonies. In a colony there is usually one queen, many worker bees, and drones. Worker bees collect nectar and pollen from flowers. They use the nectar to produce honey. Bees also help plants grow by pollinating flowers.'

# Picking a task

For this section I picked a task that I'd like to have GPT do.
1. Wrote a short description of what task I tried, why I was interested in it.
2. Gave some code so viewers can reproduce what I did via an Open API call.  Included is an output of my code pasted into a text/markdown cell—outputs which can vary across runs, incase viewers want to see the specific outputs I'm referring to.
3. Wrote a short qualitative analysis of whether or not GPT did the task well.

In [ ]:
# I tried a keyword extraction task.
#I was interested in it because it is a simple way to test whether GPT can identify the most important ideas in a short passage without needing examples.
def extract_keywords(article_text):
  client = openai.OpenAI()

  response = client.chat.completions.create(
      model="gpt-5-mini",
      messages=[
          {
              "role": "system",
              "content": "Read the article and return exactly 5 important keywords or short key phrases separated by commas."
          },
          {
              "role": "user",
              "content": article_text
          }
      ],
      top_p=1,
      frequency_penalty=0,
      presence_penalty=0
  )

  keywords = response.choices[0].message.content.strip()
  return keywords

article_text = """
The honey bee is a social insect that lives in colonies. A colony usually includes one queen, many worker bees, and drones. Worker bees collect nectar and pollen from flowers. They use nectar to make honey. Bees also help plants grow by pollinating flowers.
"""

print(extract_keywords(article_text))



honey bee, colonies, queen, worker bees, pollination


Short paragraph giving qualitative analysis of how well GPT did for this task.





In [ ]:
#Example output:
#honey bee, colonies, nectar, honey, pollinating flowers

#GPT did this task well. The keywords it returned matched the main ideas in the passage and left out less important details. The wording could vary across runs, but the output was still relevant and useful. Overall, GPT handled this zero-shot keyword extraction task effectively.


*Edit this cell to write your short paragraph here.*

# In-Context Learning

In addition to zero-shot learning explored earlier, large language models can exploit **in-context learning (ICL)** — the ability to adapt to a new task purely from examples shown inside the prompt, without any gradient updates.

This stands in contrast to *fine-tuning*, where the model's weights are permanently updated on training data.  ICL is:
- **Cheaper**: no training API calls or GPU time.
- **Faster**: no waiting for a training job to finish.
- **Flexible**: swap out examples at inference time.

In this section I will implement a **biography parser**: given a free-text Wikipedia biography, extract structured attributes in key-value format. I will then compare a **zero-shot** version (no examples in the prompt) with a **few-shot** version (several demonstrations in the prompt) using the same precision / recall / F-score metrics that would have been used to grade a fine-tuned model.

## The Task

Given a biography like:

> Jill Tracy Jacobs Biden (born June 3, 1951) is an American educator and the current first lady of the United States as the wife of President Joe Biden...

Parser should output structured attributes:

```
notable_type: First Lady of the United States
name: Jill Biden
gender: female
nationality: American
birth_date: 03 June 1951
birth_place: Hammonton, New Jersey
alma_mater: University of Delaware
occupation: professor of English at Northern Virginia Community College
partner: Joe Biden
children: Ashley Biden, Beau Biden (stepson), Hunter Biden (stepson)
```

The dataset is from [SynthBio: A Case Study in Human-AI Collaborative Curation of Text Datasets](https://www.cis.upenn.edu/~ccb/publications/synthbio.pdf) (Yuan et al., NeurIPS 2021).

## Load the Data

In [ ]:
!wget https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_train.json

# If you're running this locally on a Mac, you can use the following command instead.
# !curl -o SynthBio_train.json https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_train.json

--2026-04-14 05:00:25--  https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_train.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5807118 (5.5M) [text/plain]
Saving to: ‘SynthBio_train.json.1’

SynthBio_train.json 100%[===================>]   5.54M  --.-KB/s    in 0.08s   

2026-04-14 05:00:25 (69.3 MB/s) - ‘SynthBio_train.json.1’ saved [5807118/5807118]



In [ ]:
import json
import random

def load_wiki_bio_data(filename='SynthBio_train.json', num_bios=100, randomized=True):
    with open(filename) as f:
        synth_bio_data = json.load(f)
    if randomized:
        random.shuffle(synth_bio_data)
    bios = []
    for data in synth_bio_data:
        notable_type = data['notable_type']
        attributes = "notable_type: {notable_type} | {other_attributes}".format(
            notable_type=notable_type,
            other_attributes=data['serialized_attrs']
        )
        biography = data['biographies'][0]
        # attrs dict for use as gold standard
        attrs_dict = data.get('attrs', {})
        attrs_dict['notable_type'] = notable_type
        bios.append((biography, attributes.replace(" | ", "\n"), attrs_dict))
    return bios[:min(num_bios, len(bios))]

wiki_bios = load_wiki_bio_data()
bio_example, attrs_example, attrs_dict_example = wiki_bios[0]
print("=== BIOGRAPHY ===")
print(bio_example[:500])
print("\n=== ATTRIBUTES (structured) ===")
print(attrs_example[:500])

=== BIOGRAPHY ===
Esmerelda Gautier was born on August 21, 1944 in Paris, France. They was the daughter of Maria Gautier and Jean Gautier and was married to Alan Tice. Esmerelda Gautier know for the discovery of the universe's first conscious artificial intelligence. They were attended to Ecole Normale Superieure de Percy. A thesis with the title of 2019 and institutions Ecole Normale Superieure, Paris, France; University of Chicago, USA. They were influences daVinci and also influenced in Henrique Lemes, Mandy K

=== ATTRIBUTES (structured) ===
notable_type: scientist
name: Esmerelda Gautier
gender: female
birth_date: 21 August 1944
birth_place: Paris, France
occupation: quantum biologist & philosopher
fields: artificial intelligence
known_for: the discovery of the universe's first conscious artificial intelligence
hometown: Paris, France
nationality: French
citizenship: French and American
alma_mater: Ecole Normale Superieure de Percy
thesis_title: Instituting the Cosmos in the Mind


In [ ]:
!wget https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_test.json

# If you're running this locally on a Mac, you can use the following command instead.
# !curl -o SynthBio_test.json https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_test.json

--2026-04-14 05:00:29--  https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 665457 (650K) [text/plain]
Saving to: ‘SynthBio_test.json.1’

SynthBio_test.json. 100%[===================>] 649.86K  --.-KB/s    in 0.04s   

2026-04-14 05:00:29 (14.9 MB/s) - ‘SynthBio_test.json.1’ saved [665457/665457]



In [ ]:
import json

def load_wiki_bio_test_set(filename='SynthBio_test.json', max_test_items=10, randomized=True):
    """
    Loads our wikibio test set, returns a list of (biography_text, gold_attributes_dict) tuples.
    """
    with open(filename) as f:
        synth_bio_data = json.load(f)
    if randomized:
        random.shuffle(synth_bio_data)
    bios = []
    for data in synth_bio_data:
        notable_type = data['notable_type']
        attributes = data['attrs']
        attributes['notable_type'] = notable_type
        biography = data['biographies'][0]
        bios.append((biography, attributes))
    return bios[:min(max_test_items, len(bios))]

def convert_to_dict(predicted_attributes_txt):
    """
    Converts predicted attributes from text format (key: value per line) into a dictionary.
    """
    predicted_attributes = {}
    for line in predicted_attributes_txt.split('\n'):
        try:
            attribute, value = line.split(':', 1)
            predicted_attributes[attribute.strip()] = value.strip()
        except ValueError:
            continue
    return predicted_attributes

## Precision, Recall and F-score

We use the same evaluation framework as would be used for a fine-tuned model. Each attribute is scored independently, then averaged.

In [ ]:
from collections import Counter

def update_counts(gold_attributes, predicted_attributes,
                  true_positives, false_positives, false_negatives, all_attributes):
    # True positives and false negatives
    for attribute in gold_attributes:
        all_attributes[attribute] += 1
        if attribute in predicted_attributes:
            predicted_values = predicted_attributes[attribute].split(',')
            gold_values = gold_attributes[attribute].split(',')
            for value in gold_values:
                if value.strip() in predicted_values:
                    true_positives[attribute] += 1
                else:
                    false_negatives[attribute] += 1
        else:
            false_negatives[attribute] += 1

    # False positives
    for attribute in predicted_attributes:
        if attribute not in gold_attributes:
            all_attributes[attribute] += 1
            false_positives[attribute] += 1
        else:
            gold_values = gold_attributes[attribute].split(',')
            predicted_values = predicted_attributes[attribute].split(',')
            for value in predicted_values:
                if value.strip() not in gold_values:
                    false_positives[attribute] += 1


def evaluate_parser(parse_fn, wiki_bio_test, threshold_count=5):
    """
    Evaluate a parsing function on the test set.
    parse_fn: callable that takes a biography string and returns an attribute string.
    Returns (average_precision, average_recall, average_fscore).
    """
    true_positives  = Counter()
    false_positives = Counter()
    false_negatives = Counter()
    all_attributes  = Counter()

    for bio, gold_attributes in wiki_bio_test:
        raw = parse_fn(bio)
        predicted_attributes = convert_to_dict(raw)
        update_counts(gold_attributes, predicted_attributes,
                      true_positives, false_positives, false_negatives, all_attributes)

    average_precision = 0.0
    average_recall    = 0.0
    total = 0

    for attribute in all_attributes:
        if all_attributes[attribute] < threshold_count:
            continue
        tp = true_positives[attribute]
        fp = false_positives[attribute]
        fn = false_negatives[attribute]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fscore    = (precision + recall) / 2
        print(f"{attribute.upper():<30}  P={precision:.2f}  R={recall:.2f}  F={fscore:.2f}")
        average_precision += precision
        average_recall    += recall
        total += 1

    average_precision /= total if total > 0 else 1
    average_recall    /= total if total > 0 else 1
    average_fscore     = (average_precision + average_recall) / 2

    print("---")
    print(f"{'AVERAGE':<30}  P={average_precision:.2f}  R={average_recall:.2f}  F={average_fscore:.2f}")
    return average_precision, average_recall, average_fscore

## Set up the OpenAI API

In [ ]:
%%capture
%pip install --upgrade openai

In [ ]:
import os
import openai
from getpass import getpass

print('Enter OpenAI API key:')
openai.api_key = getpass()
os.environ['OPENAI_API_KEY'] = openai.api_key

Enter OpenAI API key:
··········


## Part 1 — Zero-Shot Bio Parser

Implements `parse_bio_zero_shot(biography)`.

The function should call the OpenAI API with **no demonstration examples** — only a system instruction and the biography text.  The model must figure out the output format entirely from the instruction.

- Explicit in system prompt about the expected output format (`key: value`, one per line).
- Ask the model to only output the structured attributes, nothing else.

In [ ]:
import openai
import os
import time

def parse_bio_zero_shot(biography):
    """
    Zero-shot biography parser.
    Given a free-text biography, returns a string of key: value attributes (one per line).
    No demonstration examples are included in the prompt.
    """
    client = openai.OpenAI()
    # write system prompt
    system_prompt = """Extract structured attributes from the biography.

Return ONLY lines in this format:
key: value

Rules:
- One attribute per line
- Use lowercase snake_case keys
- Extract as many attributes as possible
- Include likely fields such as name, birth_date, death_date, birth_place, nationality, occupation, instrument, mother, father, partner, children, and notable_type when available
- Do not include explanations or extra text
    """

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "developer", "content": system_prompt},
            {"role": "user",   "content": biography},
        ],

    )
    reasoning_effort="low",
    max_completion_tokens=1200,
    return response.choices[0].message.content.strip()


# Quick sanity check
test_bio = wiki_bios[0][0]
print("=== INPUT BIOGRAPHY (first 300 chars) ===")
print(test_bio[:300])
print("\n=== ZERO-SHOT OUTPUT ===")
print(parse_bio_zero_shot(test_bio))

=== INPUT BIOGRAPHY (first 300 chars) ===
Esmerelda Gautier was born on August 21, 1944 in Paris, France. They was the daughter of Maria Gautier and Jean Gautier and was married to Alan Tice. Esmerelda Gautier know for the discovery of the universe's first conscious artificial intelligence. They were attended to Ecole Normale Superieure de 

=== ZERO-SHOT OUTPUT ===
name: Esmerelda Gautier
birth_date: 1944-08-21
birth_place: Paris, France
mother: Maria Gautier
father: Jean Gautier
partner: Alan Tice
nationality: French
citizenship: French; American
occupation: artificial_intelligence_researcher
fields: artificial intelligence
alma_mater: Ecole Normale Superieure de Percy; Ecole Normale Superieure, Paris, France; University of Chicago, USA
thesis_title: 2019
thesis_year: 2019
thesis_institutions: Ecole Normale Superieure, Paris, France; University of Chicago, USA
influences: daVinci
influenced: Henrique Lemes; Mandy Klein
notable_type: discovery_of_the_universe_first_conscious_artificia

## Part 2 — Few-Shot Bio Parser

This implements `parse_bio_few_shot(biography, examples, k=5)`.

This time I include **k demonstration examples** drawn from the training set directly inside the prompt.  Each demonstration shows the model:
1. A biography (input)
2. The correct structured attributes (expected output)

This is the core idea of **in-context learning**: the model sees the input→output pattern repeated k times, then is asked to apply it to a new input.

- Format each example consistently (e.g., `Biography: ...\nAttributes:\n...`).
- Separate demonstrations from the final query clearly (e.g., with `---`).
- Experiment with different values of k (1, 3, 5, 10).

In [ ]:
def build_few_shot_prompt(biography, examples, k=5):
    """
    Constructs a few-shot prompt string.
    examples: list of (biography_text, attributes_text, attrs_dict) from load_wiki_bio_data()
    k: number of demonstrations to include
    Returns the full prompt string (user turn content).
    """
    # build the prompt
    # Includes k examples from `examples`, then the query biography.
    prompt = ""
    selected_examples = examples[:k]

    for bio_text, attrs_text, _ in selected_examples:
        prompt += "Biography:\n"
        prompt += bio_text.strip() + "\n\n"
        prompt += "Attributes:\n"
        prompt += attrs_text.strip() + "\n\n"
        prompt += "###\n\n"

    prompt += "Biography:\n"
    prompt += biography.strip() + "\n\n"
    prompt += "Attributes:\n"
    return prompt


def parse_bio_few_shot(biography, examples, k=5):
    """
    Few-shot biography parser.
    Given a free-text biography and a pool of training examples,
    returns a string of key: value attributes (one per line).
    """
    client = openai.OpenAI()
    # TODO - write system prompt (can reuse or adapt the zero-shot one)
    system_prompt = """Extract structured attributes from biographies.

Return ONLY lines in this format:
key: value

Rules:
- One attribute per line
- Use lowercase snake_case keys
- Follow the same format as the examples
- Extract as many attributes as possible
- Do not include explanations or extra text
"""

    prompt = build_few_shot_prompt(biography, examples, k)

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "developer", "content": system_prompt},
            {"role": "user",   "content": prompt},
        ],
        reasoning_effort="low",
        max_completion_tokens=1200,
    )
    return response.choices[0].message.content.strip()


# Quick sanity check
print("=== INPUT BIOGRAPHY (first 300 chars) ===")
print(test_bio[:300])
print("\n=== FEW-SHOT OUTPUT (k=5) ===")
# Use the remaining training bios (excluding index 0) as the example pool
example_pool = wiki_bios[1:]
print(parse_bio_few_shot(test_bio, example_pool, k=5))

=== INPUT BIOGRAPHY (first 300 chars) ===
Esmerelda Gautier was born on August 21, 1944 in Paris, France. They was the daughter of Maria Gautier and Jean Gautier and was married to Alan Tice. Esmerelda Gautier know for the discovery of the universe's first conscious artificial intelligence. They were attended to Ecole Normale Superieure de 

=== FEW-SHOT OUTPUT (k=5) ===
notable_type: scientist
name: Esmerelda Gautier
gender: female
birth_date: 21 August 1944
birth_place: Paris, France
mother: Maria Gautier
father: Jean Gautier
spouse: Alan Tice
known_for: discovery of the universe's first conscious artificial intelligence
alma_mater: Ecole Normale Superieure de Percy
education_institutions: Ecole Normale Superieure, Paris, France; University of Chicago, USA
thesis_title: 2019
nationality: French
citizenship: French; American
fields: artificial intelligence
influences: daVinci
influenced: Henrique Lemes, Mandy Klein


## Part 3 — Evaluate and Compare
Run both parsers on the test set and compare their precision, recall, and F-score.


Comparing zero-shot vs. few-shot lets you measure how much the in-context demonstrations help.

In [ ]:
testset_filename = 'SynthBio_test.json'
max_test_items = 50   # set to at least 50 for final submission

wiki_bio_test = load_wiki_bio_test_set(testset_filename, max_test_items)
example_pool  = wiki_bios   # full training set as the ICL example pool

# --- Zero-shot ---
print("=" * 60)
print("ZERO-SHOT PARSER")
print("=" * 60)
zs_p, zs_r, zs_f = evaluate_parser(
    lambda bio: parse_bio_zero_shot(bio),
    wiki_bio_test,
    threshold_count=5
)

ZERO-SHOT PARSER
NAME                            P=0.74  R=0.74  F=0.74
GENDER                          P=0.50  R=0.06  F=0.28
NATIONALITY                     P=0.73  R=0.70  F=0.71
BIRTH_DATE                      P=0.30  R=0.36  F=0.33
BIRTH_PLACE                     P=0.44  R=0.44  F=0.44
DEATH_DATE                      P=0.25  R=0.21  F=0.23
DEATH_CAUSE                     P=0.86  R=0.18  F=0.52
START_AGE                       P=0.00  R=0.00  F=0.00
NOTABLE_ASCENTS                 P=0.13  R=0.11  F=0.12
PARTNERSHIPS                    P=0.00  R=0.00  F=0.00
MOTHER                          P=0.74  R=0.64  F=0.69
FATHER                          P=0.80  R=0.76  F=0.78
PARTNER                         P=0.73  R=0.63  F=0.68
CHILDREN                        P=0.30  R=0.25  F=0.28
NOTABLE_TYPE                    P=0.19  R=0.22  F=0.20
OCCUPATION                      P=0.09  R=0.15  F=0.12
SPOUSE                          P=0.00  R=0.00  F=0.00
BIRTH_NAME                      P=0.67  R=0.25  

In [ ]:
# --- Few-shot ---
print("=" * 60)
print("FEW-SHOT PARSER  (k=5)")
print("=" * 60)
fs_p, fs_r, fs_f = evaluate_parser(
    lambda bio: parse_bio_few_shot(bio, example_pool, k=5),
    wiki_bio_test,
    threshold_count=5
)

FEW-SHOT PARSER  (k=5)
NAME                            P=0.78  R=0.78  F=0.78
GENDER                          P=0.74  R=0.50  F=0.62
NATIONALITY                     P=0.83  R=0.78  F=0.80
BIRTH_DATE                      P=0.81  R=0.78  F=0.80
BIRTH_PLACE                     P=0.44  R=0.46  F=0.45
DEATH_DATE                      P=0.48  R=0.33  F=0.41
DEATH_CAUSE                     P=0.77  R=0.68  F=0.72
START_AGE                       P=0.83  R=0.50  F=0.67
NOTABLE_ASCENTS                 P=0.10  R=0.10  F=0.10
PARTNERSHIPS                    P=0.50  R=0.10  F=0.30
MOTHER                          P=0.90  R=0.74  F=0.82
FATHER                          P=0.93  R=0.80  F=0.86
PARTNER                         P=0.84  R=0.46  F=0.65
CHILDREN                        P=0.35  R=0.26  F=0.31
NOTABLE_TYPE                    P=0.59  R=0.58  F=0.59
HOMETOWN                        P=0.23  R=0.17  F=0.20
BIRTH_NAME                      P=0.50  R=0.12  F=0.31
ALIAS                           P=0.20  R=

## Analysis

In [ ]:


# Zero-shot results
zero_shot_precision = 0.39
zero_shot_recall    = 0.23
zero_shot_fscore    = 0.31

# Few-shot results
few_shot_precision = 0.43
few_shot_recall    = 0.28
few_shot_fscore    = 0.35

print(f"Zero-shot  — P: {zero_shot_precision:.2f}  R: {zero_shot_recall:.2f}  F: {zero_shot_fscore:.2f}")
print(f"Few-shot   — P: {few_shot_precision:.2f}  R: {few_shot_recall:.2f}  F: {few_shot_fscore:.2f}")
print(f"F-score improvement from ICL: {few_shot_fscore - zero_shot_fscore:+.2f}")

# What attributes had the highest F-scores in the few-shot setting?
best_attributes = {
    "attribute_name1": 0.78,
    "attribute_name2": 0.86,
    "attribute_name3": 0.82,
}

# What attributes had the lowest F-scores in the few-shot setting?
worst_attributes = {
    "attribute_name1": 0.08,
    "attribute_name2": 0.16,
    "attribute_name3": 0.12,
}

Zero-shot  — P: 0.39  R: 0.23  F: 0.31
Few-shot   — P: 0.43  R: 0.28  F: 0.35
F-score improvement from ICL: +0.04
